# Lesson 1: Simple ReAct Agent from Scratch

In [ ]:
# based on https://til.simonwillison.net/llms/python-react-pattern

In [ ]:
from dotenv import load_dotenv
from utils import printer
_ = load_dotenv()

In [ ]:
import re
from openai import OpenAI

In [ ]:
ollama_service = "http://localhost:11434/v1"
model_name = "qwen3-vl:4b-instruct"

In [ ]:
client = OpenAI(base_url=ollama_service)

In [ ]:
chat_completion = client.chat.completions.create(
    model=model_name,
    messages=[{"role": "user", "content": "Hello world"}]
)

In [ ]:
print(chat_completion.choices[0].message.content)

In [ ]:
class Agent:
    def __init__(self, system_message=""):
        self.messages = []

        if system_message:
            self.messages.append({"role": "system", "content": system_message})


    def __call__(self, message):
        self.messages.append(message)
        result = self.execute()
        self.messages.append({"role": "assistant", "content": result})
        return result

    def execute(self):
        completion = client.chat.completions.create(
                        model=model_name,
                        temperature=0,
                        messages=self.messages)
        return completion.choices[0].message.content
    

In [ ]:
SYSTEM_MESSAGE = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:

calculate:
e.g. calculate: 4 * 7 / 3
Runs a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary

average_dog_weight:
e.g. average_dog_weight: Collie
returns average weight of a dog when given the breed

Example session:

Question: How much does a Bulldog weigh?
Thought: I should look the dogs weight using average_dog_weight
Action: average_dog_weight: Bulldog
PAUSE

You will be called again with this:

Observation: A Bulldog weights 51 lbs

You then output:

Answer: A bulldog weights 51 lbs
""".strip()

In [ ]:
def calculate(what):
    return eval(what)

def average_dog_weight(name):
    if name in "Scottish Terrier": 
        return("Scottish Terriers average 20 lbs")
    elif name in "Border Collie":
        return("a Border Collies average weight is 37 lbs")
    elif name in "Toy Poodle":
        return("a toy poodles average weight is 7 lbs")
    else:
        return("An average dog weights 50 lbs")

tools_mapping = {
    "calculate": calculate,
    "average_dog_weight": average_dog_weight
}

In [ ]:
agent = Agent(SYSTEM_MESSAGE)

In [ ]:
message = "How much does a toy poodle weigh?"

response = agent({"role": "user", "content": message})
print(response)

In [ ]:
def parse_actions(response):
    action_re = re.compile(r"^Action: (\w+): (.*)$")
    actions_match = [action_re.match(a) for a in response.split("\n")]

    actions = [
        {
            "name": action.group(1),
            "arg": action.group(2)
        } for action in actions_match if action is not None
    ]

    return actions


def get_observation(response):

    if not (actions := parse_actions(response)):
        return None

    # for simplicity we just take the first action
    action = actions.pop()
    action_name = action["name"]
    tool = tools_mapping.get(action_name)
    
    if tool:
        arg = action["arg"]

        print(f"Invoke: {action_name}(\"{arg}\")")
        observation_message = tool(arg)
    else:
        observation_message = f"Unknown action {action_name}"

    tool_message = f"Observation: {observation_message}"
    print(tool_message)

    return {"role": "tool", "content": tool_message}

In [ ]:
observation = get_observation(response)

In [ ]:
agent(observation)

In [ ]:
printer(agent.messages)

```mermaid
flowchart TD
    START --> system
    system --> user
    user -- task --> assistant
    assistant -- action --> tool
    tool -- observation --> assistant
    assistant -- answer --> END
```

In [ ]:
agent = Agent(SYSTEM_MESSAGE)

In [ ]:
question = """I have 2 dogs, a border collie and a scottish terrier. \
What is their combined weight"""

response = agent({"role": "user", "content": question})
print(response)

In [ ]:
observation = get_observation(response)

In [ ]:
response = agent(observation)
print(response)

In [ ]:
observation = get_observation(response)

In [ ]:
response = agent(observation)
print(response)

In [ ]:
observation = get_observation(response)

In [ ]:
agent(observation)

### Add loop 

In [ ]:
def query(input):
    agent = Agent(SYSTEM_MESSAGE)
    next_prompt = {"role": "user", "content": input}

    while True:
        response = agent(next_prompt)
        print(f"\n{response}\n")

        if not (observation := get_observation(response)):
            break

        next_prompt = observation


In [ ]:
question = """I have 2 dogs, a border collie and a scottish terrier. \
What is their combined weight"""

query(question)